In [ ]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Image
from scipy.optimize import least_squares
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse
from matplotlib import collections as mc
import mpl_toolkits.mplot3d.art3d as art3d
import json
from skimage.measure import EllipseModel
import math

In [ ]:
def plot_model(values, bike_params):

    subs = {
        phi: values[0],
        theta: values[1],
        psi: values[2],
        delta: values[3],
        x_r: values[4], 
        y_r: values[5],
        z_r: values[6],
        lr: bike_params['lr'], 
        lf1: bike_params['lf1'],
        lf2: bike_params['lf2']
    }


    pos_Cf_x = float(Cf.pos_from(O).express(N).dot(N.x).subs(subs).evalf())
    pos_Cf_y = float(Cf.pos_from(O).express(N).dot(N.y).subs(subs).evalf())
    pos_Cf_z = float(Cf.pos_from(O).express(N).dot(N.z).subs(subs).evalf())
    pos_Cf = [pos_Cf_x, pos_Cf_y, pos_Cf_z]

    pos_Cr_x = float(Cr.pos_from(O).express(N).dot(N.x).subs(subs).evalf())
    pos_Cr_y = float(Cr.pos_from(O).express(N).dot(N.y).subs(subs).evalf())
    pos_Cr_z = float(Cr.pos_from(O).express(N).dot(N.z).subs(subs).evalf())
    pos_Cr = [pos_Cr_x, pos_Cr_y, pos_Cr_z]

    pos_S_x = float(S.pos_from(O).express(N).dot(N.x).subs(subs).evalf())
    pos_S_y = float(S.pos_from(O).express(N).dot(N.y).subs(subs).evalf())
    pos_S_z = float(S.pos_from(O).express(N).dot(N.z).subs(subs).evalf())
    pos_S = [pos_S_x, pos_S_y, pos_S_z]

    pos_P_x = float(P.pos_from(O).express(N).dot(N.x).subs(subs).evalf())
    pos_P_y = float(P.pos_from(O).express(N).dot(N.y).subs(subs).evalf())
    pos_P_z = float(P.pos_from(O).express(N).dot(N.z).subs(subs).evalf())
    pos_P = [pos_P_x, pos_P_y, pos_P_z]


    points = np.array([
        pos_Cr,
        pos_S,
        pos_P,
        pos_Cf
    ])

    points_2d_x = np.array([
        pos_Cr_x,
        pos_S_x,
        pos_P_x,
        pos_Cf_x
    ])

    points_2d_z = np.array([
        pos_Cr_z,
        pos_S_z,
        pos_P_z,
        pos_Cf_z
    ])

    x, y, z = points.T

    plt.plot(points_2d_x, points_2d_z)
    plt.gca().set_aspect('equal')
    plt.scatter(pos_Cr_x, pos_Cr_z, color = 'blue')
    plt.scatter(pos_Cf_x, pos_Cf_z, color = 'green')
    plt.xlim(0, 1920)
    plt.ylim(0, 1080)
    plt.show()


    fig = plt.figure()
    ax = plt.axes(projection='3d')
    # ax.set_aspect('equal')
    ax.set_box_aspect(aspect=(1, 1, 1))
    ax.axes.set_xlim3d(0, 500)
    ax.axes.set_ylim3d(-500,500)
    ax.axes.set_zlim3d(300, 500)
    
    ax.view_init(elev=12, azim=120, roll=0)
    ax.plot(x, y, z, color='tab:orange', lw=2)
    ax.scatter(pos_Cr_x, pos_Cr_y, pos_Cr_z, color = 'blue')
    ax.scatter(pos_Cf_x, pos_Cf_y, pos_Cf_z, color = 'green')

    ax.set(xlabel = 'x', ylabel = 'y', zlabel = 'z')
    plt.show()

    return 

In [ ]:
phi, theta, psi, delta = sp.symbols('varphi, theta, psi, delta')
lr, lf1, lf2 = sp.symbols('l_r, l_f1, l_f2')
x_r, y_r, z_r = sp.symbols('x_r, y_r, z_r')

N, R, Rs, F = sp.symbols('N, R, Rs, F', cls=me.ReferenceFrame)


R.orient_body_fixed(N, (psi, phi, theta), 'zxy')
F.orient_axis(R, delta, R.z)

Cr = me.Point('C_r')
Cf = me.Point('C_f')
S = me.Point('S')
P = me.Point('P')
O = me.Point('O')

O.set_pos(O, 0)
O.set_vel(N, 0)

Cr.set_pos(O, x_r*N.x + y_r*N.y + z_r*N.z) # assuming that the bicycle is never in front of the set origin.
S.set_pos(Cr, lr*R.x)
P.set_pos(S, -lf1 * F.z)
Cf.set_pos(P, lf2 * F.x)

Cf.pos_from(Cr)

In [ ]:
Ry_xz_x = R.y.express(N).dot(N.x)
Ry_xz_z = R.y.express(N).dot(N.z) 

Fy_xz_x = F.y.express(N).dot(N.x)
Fy_xz_z = F.y.express(N).dot(N.z) 

r_Cf_Cr_xz_x = Cf.pos_from(Cr).dot(N.x)
r_Cf_Cr_xz_z = Cf.pos_from(Cr).dot(N.z)

r_Cr_O_xz_x = Cr.pos_from(O).dot(N.x)
r_Cr_O_xz_z = Cr.pos_from(O).dot(N.z)

r_P_S_x = R.x.dot(N.x)
r_P_S_z = R.z.dot(N.z)


eval_f1 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), Ry_xz_x)
eval_f2 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), Ry_xz_z)

eval_f3 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), Fy_xz_x)
eval_f4 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), Fy_xz_z)

eval_f5 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_Cf_Cr_xz_x)
eval_f6 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_Cf_Cr_xz_z)
eval_f7 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_Cr_O_xz_x)
eval_f8 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_Cr_O_xz_z)
eval_f9 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_P_S_x)
eval_f10 = sp.lambdify((phi, theta, psi, delta, x_r, y_r, z_r, lr, lf1, lf2), r_P_S_z)


In [ ]:
bike_params = {
    'lr': 140,
    'lf1': 50,
    'lf2': 10
}

bike_params2 = [140, 50, 10]

# Bounds for least squares

# lb = [-np.pi/2, -np.pi, -np.pi, -np.pi, 0, -np.inf, 0]
# ub = [np.pi/2, np.pi, np.pi, np.pi, 1920, np.inf, 1080]

lb = [-np.pi/2, -np.pi/2, -np.pi/2, -np.pi/2, 0, -np.inf, 0]
ub = [np.pi/2, np.pi/2, np.pi/2, np.pi/2, 1920, np.inf, 1080]

In [ ]:
imgdata_gopro_f417 = {'Ry_x': -1.1365207692126576e-05, 'Ry_z': 0.2568298606099628, 'Fy_x': -8.196223406115411e-07, 'Fy_z': 0.2613723913311994, 'r_Cf_Cr_x': 181.99033660600298, 'r_Cf_Cr_z': -0.9725647864910343, 'r_Cr_O_x': 224.00106398940443, 'r_Cr_O_z': 365.9684410278786, 'r_P_S_x': 0.9999974861660557, 'r_P_S_z': 0.002164085566933416}


imgdata_gopro_f390 = {'Ry_x': -0.15210617129358794, 'Ry_z': 0.9668809907320831, 'Fy_x': -0.07918322569728496, 'Fy_z': 0.5972328827166676, 'r_Cf_Cr_x': 41.96430105312106, 'r_Cf_Cr_z': -22.948570828897232, 'r_Cr_O_x': 4.0462574530756115, 'r_Cr_O_z': 382.96454224288823, 'r_P_S_x': -0.986901460479248, 'r_P_S_z': 0.15985709167405412}


In [ ]:
def residual_eqs(x, data, bike_params):

    subs = (x[0], x[1], x[2], x[3], x[4], x[5], x[6], bike_params['lr'],
            bike_params['lf1'], bike_params['lf2'])

    
    f1 = eval_f1(*subs) - data['Ry_x']
    f2 = eval_f2(*subs) - data['Ry_z']
    f3 = eval_f3(*subs) - data['Fy_x']
    f4 = eval_f4(*subs) - data['Fy_z']
    f5 = eval_f5(*subs) - data['r_Cf_Cr_x']
    f6 = eval_f6(*subs) - data['r_Cf_Cr_z']
    f7 = eval_f7(*subs) - data['r_Cr_O_x']
    f8 = eval_f8(*subs) - data['r_Cr_O_z']
    f9 = eval_f9(*subs) - data['r_P_S_x']
    f10 = eval_f10(*subs) - data['r_P_S_z']
    
    return np.array([f1, f2, f3, f4, f5, f6, f7, f8, f9, f10])

In [ ]:
# Initial guess
x0 = np.array([
    np.deg2rad(0),
    np.deg2rad(-30), 
    np.deg2rad(0),
    np.deg2rad(0),
    180,
    0,
    400
])

# x0 = [
#     0,
#     np.deg2rad(-10), 
#     np.deg2rad(75),
#     np.deg2rad(25),
#     0,
#     0,
#     400
# ]

result = least_squares(residual_eqs, x0, args = (imgdata_gopro_f417, bike_params), bounds = (lb, ub))

result

In [ ]:
print('phi      = ', np.rad2deg(result['x'][0]))
print('theta    = ', np.rad2deg(result['x'][1]))
print('psi      = ', np.rad2deg(result['x'][2]))
print('delta    = ', np.rad2deg(result['x'][3]))
print('x        = ', result['x'][4])
print('y        = ', result['x'][5])
print('z        = ', result['x'][6])

plot_model(result['x'], bike_params)

In [ ]:
with open('/home/eimolgon/Documents/PhD-Project/vid2dyn/output/data4model_gopro.json') as json_file:
    data = json.load(json_file)

results_hist = []

imgdata_temp = {
    'Ry_x': 0, 
    'Ry_z': 0,
    'Fy_x': 0, 
    'Fy_z': 0,
    'r_Cf_Cr_x': 0, 
    'r_Cf_Cr_z': 0, 
    'r_Cr_O_x': 0, 
    'r_Cr_O_z': 0,
    'r_P_S_x': 0,
    'r_P_S_z': 0
    }

# Initial guess
# x0 = [
#     0,
#     np.deg2rad(-10), 
#     np.deg2rad(75),
#     np.deg2rad(25),
#     0,
#     0,
#     400
# ]

x0 = [
    np.deg2rad(0),
    np.deg2rad(-30), 
    np.deg2rad(0),
    np.deg2rad(0),
    180,
    0,
    400
]


for i in range(27, 28):

    imgdata_temp['Ry_x'] = data['Ry_x'][i]
    imgdata_temp['Ry_z'] = data['Ry_z'][i]

    imgdata_temp['Fy_x'] = data['Fy_x'][i]
    imgdata_temp['Fy_z'] = data['Fy_z'][i]

    imgdata_temp['r_Cf_Cf_x'] = data['r_Cf_Cr_x'][i]
    imgdata_temp['r_Cf_Cr_z'] = data['r_Cf_Cr_z'][i]
    imgdata_temp['r_Cr_O_x'] = data['r_Cr_O_x'][i]
    imgdata_temp['r_Cr_O_z'] = data['r_Cr_O_z'][i]
    imgdata_temp['r_P_S_x'] = data['r_P_S_x'][i]
    imgdata_temp['r_P_S_z'] = data['r_P_S_z'][i]


    # result_iter = least_squares(residual_eqs, x0, args = (imgdata_temp, 1))
    result_iter = least_squares(residual_eqs, x0, args = (imgdata_temp, bike_params), bounds = (lb, ub))
    results_hist.append(result_iter)

    x0 = result_iter['x']

for j, k in zip(imgdata_gopro_f417.values(), imgdata_temp.values()):
    print(j - k)    

# From here is the best result, check

In [ ]:
def readEllipse(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    ellipses = []

    for line in lines:
        parts = list(map(float, line.strip().split()))

        class_id = int(parts[0])
        points = np.array(parts[1:]).reshape(-1, 2)
        center = np.mean(points, axis=0)
        centered_points = points - center
        cov = np.cov(centered_points.T)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        major_axis = 2 * math.sqrt(max(eigenvalues))
        minor_axis = 2 * math.sqrt(min(eigenvalues))
        angle = math.degrees(math.atan2(eigenvectors[1, 0], eigenvectors[0, 0]))

        ellipses.append({
            'class_id': class_id,
            'center': center,
            'major_axis': major_axis,
            'minor_axis': minor_axis,
            'angle': angle,
            'points': points
        })

    return ellipses

# with open('data4model_gopro_check.json') as json_file:
#     data = json.load(json_file)

results_hist2 = []

imgdata_temp = {
    'Ry_x': [], 
    'Ry_z': [],
    'Fy_x': [], 
    'Fy_z': [],
    'r_Cf_Cr_x': [], 
    'r_Cf_Cr_z': [], 
    'r_Cr_O_x': [], 
    'r_Cr_O_z': [],
    'r_P_S_x': [],
    'r_P_S_z': []
    }

# Initial guess
# x0 = [
#     0,
#     np.deg2rad(-10), 
#     np.deg2rad(75),
#     np.deg2rad(25),
#     0,
#     0,
#     400
# ]

x0 = [
    np.deg2rad(0),
    np.deg2rad(-30), 
    np.deg2rad(0),
    np.deg2rad(0),
    180,
    0,
    400
]


imgdata0 = {
        'Ry_x': 0,
        'Ry_z': 0,
        'Fy_x': 0,
        'Fy_z': 0,
        'r_Cf_Cr_x': 0,
        'r_Cf_Cr_z': 0,
        'r_Cr_O_x': 0,
        'r_Cr_O_z': 0,
        'r_P_S_x': 0,
        'r_P_S_z': 0
    }

screen_resolution = (1920, 1080)
assumed_origin = (screen_resolution[0]*0.2, screen_resolution[1]*0.2)

# for i in range(26, len(data['Ry_x'])):
for i in range(416, 491):
    file = f'/home/eimolgon/Documents/PhD-Project/02-video-data/gopro_test_1_1_resize_annotation_161025/labels/Train/frame_000{i}.txt'
    # file = '/home/eimolgon/Documents/PhD-Project/Video2Dynamics/data/yt-crash-005/labels/train/frame_000075.txt'

    ellipses = readEllipse(file)

    major_f = ellipses[0]['major_axis'] 
    minor_f = ellipses[0]['minor_axis'] 
    center_f = ellipses[0]['center']
    center_f[0] = center_f[0]*screen_resolution[0]
    center_f[1] = screen_resolution[1] * (1 - center_f[1])
    angle_f = ellipses[0]['angle']
    points_f = ellipses[0]['points']

    major_r = ellipses[1]['major_axis'] 
    minor_r = ellipses[1]['minor_axis'] 
    center_r = ellipses[1]['center']
    center_r[0] = center_r[0]*screen_resolution[0]
    center_r[1] = screen_resolution[1] * (1 - center_r[1])
    angle_r = ellipses[1]['angle']
    points_r = ellipses[1]['points']

    centers_x = [center_r[0], center_f[0]]
    centers_y = [center_r[1], center_f[1]]
    slope = (center_f[1]-center_r[1])/(center_f[0]-center_r[0])


    x_diff = center_f[0] - center_r[0]
    y_diff = center_f[1] - center_r[1]
    w_screen = np.sqrt(x_diff**2 + y_diff**2)

    projection_coefficients = np.polyfit(centers_x, centers_y, 1)
    wheelbase_projection = np.poly1d(projection_coefficients)
    x_projection = np.linspace(0, screen_resolution[0], 100)
    y_projection = wheelbase_projection(x_projection)

    ell_f = EllipseModel()
    ell_f.estimate(points_f)
    residuals_f = ell_f.residuals(points_f)
    xc_f, yc_f, a_f, b_f, theta_f = ell_f.params


    xc_f = xc_f * screen_resolution[0]
    yc_f = screen_resolution[1] * (1 - yc_f)
    a_f = a_f * screen_resolution[1]
    b_f = b_f * screen_resolution[0]

    ellipse_front = {'center':(xc_f, yc_f), 'major_axis': b_f, 'minor_axis': a_f, 'angle': theta_f}

    if a_f < b_f:
        cam_angle_f_2 = np.arccos(a_f/b_f)
    elif a_f >= b_f:
        cam_angle_f_2 = np.arccos(b_f/a_f)

    ell_r = EllipseModel()
    ell_r.estimate(points_r)
    residuals_r = ell_r.residuals(points_r)
    xc_r, yc_r, a_r, b_r, theta_r = ell_r.params


    xc_r = xc_r * screen_resolution[0]
    yc_r = screen_resolution[1] * (1 - yc_r)
    a_r = a_r * screen_resolution[1]
    b_r = b_r * screen_resolution[0]

    ellipse_rear = {'center':(xc_r, yc_r), 'major_axis': b_r, 'minor_axis': a_r, 'angle': theta_r}

    if a_r < b_r:
        cam_angle_r_2 = np.arccos(a_r/b_r)
    elif a_r >= b_r:
        cam_angle_r_2 = np.arccos(b_r/a_r)

    
    imgdata0['Ry_x'] = np.sin(cam_angle_r_2)*np.cos(theta_r)
    imgdata0['Ry_z'] = np.sin(cam_angle_r_2)*np.sin(theta_r)
    imgdata0['Fy_x'] = np.sin(cam_angle_f_2)*np.cos(theta_f)
    imgdata0['Fy_z'] = np.sin(cam_angle_f_2)*np.sin(theta_f)
    imgdata0['r_Cf_Cr_x'] = xc_f-xc_r
    imgdata0['r_Cf_Cr_z'] = yc_f-yc_r
    imgdata0['r_Cr_O_x'] = xc_r - assumed_origin[0]
    imgdata0['r_Cr_O_z'] = yc_r - assumed_origin[1]
    imgdata0['r_P_S_x'] = (np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))*np.sin(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2) + np.sin(cam_angle_r_2)*np.sin(theta_f - theta_r)*np.cos(cam_angle_f_2)*np.cos(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2)
    imgdata0['r_P_S_z'] = (np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))*np.cos(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2) - np.sin(cam_angle_r_2)*np.sin(theta_f)*np.sin(theta_f - theta_r)*np.cos(cam_angle_f_2)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2)

    imgdata_temp['Ry_x'].append(np.sin(cam_angle_r_2)*np.cos(theta_r))
    imgdata_temp['Ry_z'].append(np.sin(cam_angle_r_2)*np.sin(theta_r))
    imgdata_temp['Fy_x'].append(np.sin(cam_angle_f_2)*np.cos(theta_f))
    imgdata_temp['Fy_z'].append(np.sin(cam_angle_f_2)*np.sin(theta_f))
    imgdata_temp['r_Cf_Cr_x'].append(xc_f-xc_r)
    imgdata_temp['r_Cf_Cr_z'].append(yc_f-yc_r)
    imgdata_temp['r_Cr_O_x'].append(xc_r - assumed_origin[0])
    imgdata_temp['r_Cr_O_z'].append(yc_r - assumed_origin[1])
    imgdata_temp['r_P_S_x'].append((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))*np.sin(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2) + np.sin(cam_angle_r_2)*np.sin(theta_f - theta_r)*np.cos(cam_angle_f_2)*np.cos(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2))
    imgdata_temp['r_P_S_z'].append((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))*np.cos(theta_f)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2) - np.sin(cam_angle_r_2)*np.sin(theta_f)*np.sin(theta_f - theta_r)*np.cos(cam_angle_f_2)/np.sqrt((np.sin(cam_angle_f_2)*np.cos(cam_angle_r_2) - np.sin(cam_angle_r_2)*np.cos(cam_angle_f_2)*np.cos(theta_f - theta_r))**2 + np.sin(cam_angle_r_2)**2*np.sin(theta_f - theta_r)**2))




    # imgdata_temp['Ry_x'] = data['Ry_x'][i]
    # imgdata_temp['Ry_z'] = data['Ry_z'][i]

    # imgdata_temp['Fy_x'] = data['Fy_x'][i]
    # imgdata_temp['Fy_z'] = data['Fy_z'][i]

    # imgdata_temp['r_Cf_Cf_x'] = data['r_Cf_Cr_x'][i]
    # imgdata_temp['r_Cf_Cr_z'] = data['r_Cf_Cr_z'][i]
    # imgdata_temp['r_Cr_O_x'] = data['r_Cr_O_x'][i]
    # imgdata_temp['r_Cr_O_z'] = data['r_Cr_O_z'][i]
    # imgdata_temp['r_P_S_x'] = data['r_P_S_x'][i]
    # imgdata_temp['r_P_S_z'] = data['r_P_S_z'][i]


    # result_iter = least_squares(residual_eqs, x0, args = (imgdata_temp, 1))
    # print(imgdata0)
    result_iter2 = least_squares(residual_eqs, x0, args = (imgdata0, bike_params), bounds = (lb, ub))
    results_hist2.append(result_iter2)

    x0 = result_iter2['x'].copy()
    

In [ ]:
roll_hist = []
pitch_hist = []
yaw_hist = []
steer_hist = []
x_hist = []
y_hist = []
z_hist = []

for i in results_hist2:
    roll_hist.append(i['x'][0])
    pitch_hist.append(i['x'][1])
    yaw_hist.append(i['x'][2])
    steer_hist.append(i['x'][3])
    x_hist.append(i['x'][4])
    y_hist.append(i['x'][5])
    z_hist.append(i['x'][6])

data_sim = {'phi':roll_hist, 'theta':pitch_hist, 'psi':yaw_hist, 'delta':steer_hist, 'xr':x_hist, 'yr':y_hist, 'zr':z_hist}

plt.plot(yaw_hist, label = 'solver data')
# plt.plot(data_rw['x'], label = 'video data')
plt.xlabel('Frame')
plt.ylabel('x (pixels)')
plt.legend()
plt.show()